# AgriSetu — 50+ Class Plant Disease Model Training (Phase 2)
### End-to-End Pipeline on Google Colab (Free T4 GPU)

This notebook provides a complete automated pipeline to:
1. **Download public agricultural disease datasets** (PlantVillage, Rice, Wheat, Cassava, etc.) directly into Colab.
2. **Merge & clean into 50+ standardized disease classes** across major BRICS staple crops.
3. **Augment and balance** class samples using Albumentations.
4. **Fine-tune an EfficientNet model** with mixed precision (`torch.cuda.amp`) for maximum performance on free T4 GPU.
5. **Evaluate** model metrics (Top-1 / Top-3 Accuracy, Loss curves, Confusion Matrix).
6. **Export and Upload** `disease_model_best.pth` and `class_names.json` directly to Supabase Storage & Google Drive.

---

## 1. System Setup & GPU Verification
Ensure your Colab runtime is set to **GPU**: `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
!pip install -q timm albumentations scikit-learn matplotlib seaborn tqdm supabase

import os
import sys
import time
import json
import shutil
import random
from pathlib import Path
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision
from torchvision import transforms, datasets, models
import timm

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: GPU is not enabled! Please switch to GPU in Runtime settings.")

## 2. Dataset Acquisition from Kaggle / Direct Links

Upload your `kaggle.json` API token file to download the datasets automatically.
- If you don't have `kaggle.json`: Go to [Kaggle Account](https://www.kaggle.com/settings) -> Click **'Create New Token'** -> download file.

In [ ]:
from google.colab import files

# Directory setup
DATASETS_RAW_DIR = Path("/content/raw_datasets")
DATASET_MERGED_DIR = Path("/content/dataset_50plus")
DATASETS_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATASET_MERGED_DIR.mkdir(parents=True, exist_ok=True)

# Setup Kaggle API credentials
kaggle_config_dir = Path("/root/.kaggle")
kaggle_config_dir.mkdir(parents=True, exist_ok=True)

if not (kaggle_config_dir / "kaggle.json").exists():
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn == 'kaggle.json':
            shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
            print("kaggle.json configured successfully!")
else:
    print("kaggle.json already configured.")

In [ ]:
# Download datasets
print("1. Downloading PlantVillage Dataset (38 Classes)...")
!kaggle datasets download -d emmarex/plantvillage -p /content/raw_datasets/plantvillage --unzip --quiet

print("2. Downloading Rice Leaf Disease Dataset...")
!kaggle datasets download -d minhhuy271199/rice-disease-dataset -p /content/raw_datasets/rice --unzip --quiet

print("3. Downloading Cassava Leaf Disease Dataset...")
!kaggle datasets download -d check12/cassava-disease -p /content/raw_datasets/cassava --unzip --quiet

print("4. Downloading Wheat Leaf Disease Dataset...")
!kaggle datasets download -d yassinealouini/wheat-leaf-dataset -p /content/raw_datasets/wheat --unzip --quiet || \
!kaggle datasets download -d mlg-ulb/wheat-leaf-disease-dataset -p /content/raw_datasets/wheat --unzip --quiet || echo "Using fallback wheat dataset."

print("All raw datasets downloaded successfully!")

## 3. Dataset Merging & Standardized 50+ Class Taxonomy

We now harmonize all classes into the standard naming format: `Plant_Name___Disease_Name` (e.g. `Tomato___Late_blight`, `Rice___Brown_spot`, `Wheat___Yellow_rust`).

In [ ]:
import glob

def clean_copy_images(src_dir, target_class_name, max_samples=400):
    """Copy valid images from source directory to target class directory."""
    dest_dir = DATASET_MERGED_DIR / target_class_name
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG', '*.JPEG')
    image_files = []
    for ext in extensions:
        image_files.extend(glob.glob(f"{src_dir}/**/{ext}", recursive=True))
        
    copied = 0
    random.seed(42)
    random.shuffle(image_files)
    
    for img_path in image_files:
        if copied >= max_samples:
            break
        try:
            # Verify image is valid
            with Image.open(img_path) as im:
                im.verify()
            
            dest_path = dest_dir / f"{target_class_name}_{copied:04d}.jpg"
            shutil.copy2(img_path, dest_path)
            copied += 1
        except Exception:
            continue
            
    return copied

print("Processing PlantVillage 38 Classes...")
pv_dirs = list(Path("/content/raw_datasets/plantvillage").glob("**/*"))
pv_classes = set()
for d in pv_dirs:
    if d.is_dir() and "___" in d.name and not any(sub.is_dir() for sub in d.iterdir()):
        count = clean_copy_images(d, d.name, max_samples=350)
        if count > 10:
            pv_classes.add(d.name)
            print(f"  [PlantVillage] {d.name}: {count} images")

# Add Rice classes
print("\nProcessing Rice Disease Classes...")
rice_mapping = {
    "Bacterial_blight": "Rice___Bacterial_blight",
    "Blast": "Rice___Blast",
    "Brown_spot": "Rice___Brown_spot",
    "Tungro": "Rice___Tungro",
    "Healthy": "Rice___healthy",
    "healthy": "Rice___healthy",
    "Leaf_smut": "Rice___Leaf_smut"
}
for src_folder in Path("/content/raw_datasets/rice").glob("**/*"):
    if src_folder.is_dir():
        for key, target_name in rice_mapping.items():
            if key.lower() in src_folder.name.lower():
                count = clean_copy_images(src_folder, target_name, max_samples=350)
                if count > 10:
                    print(f"  [Rice] {target_name}: {count} images")

# Add Cassava classes
print("\nProcessing Cassava Disease Classes...")
cassava_mapping = {
    "cbb": "Cassava___Bacterial_blight",
    "cbsd": "Cassava___Brown_streak",
    "cgm": "Cassava___Green_mottle",
    "cmd": "Cassava___Mosaic_disease",
    "healthy": "Cassava___healthy"
}
for src_folder in Path("/content/raw_datasets/cassava").glob("**/*"):
    if src_folder.is_dir():
        for key, target_name in cassava_mapping.items():
            if key.lower() in src_folder.name.lower():
                count = clean_copy_images(src_folder, target_name, max_samples=350)
                if count > 10:
                    print(f"  [Cassava] {target_name}: {count} images")

# Add Wheat classes
print("\nProcessing Wheat Disease Classes...")
wheat_mapping = {
    "leaf_rust": "Wheat___Leaf_rust",
    "yellow_rust": "Wheat___Yellow_rust",
    "stripe_rust": "Wheat___Yellow_rust",
    "brown_rust": "Wheat___Leaf_rust",
    "septoria": "Wheat___Septoria",
    "healthy": "Wheat___healthy"
}
for src_folder in Path("/content/raw_datasets/wheat").glob("**/*"):
    if src_folder.is_dir():
        for key, target_name in wheat_mapping.items():
            if key.lower() in src_folder.name.lower():
                count = clean_copy_images(src_folder, target_name, max_samples=350)
                if count > 10:
                    print(f"  [Wheat] {target_name}: {count} images")

# Summary
all_classes = sorted([d.name for d in DATASET_MERGED_DIR.iterdir() if d.is_dir() and len(list(d.glob('*.jpg'))) > 0])
print("\n=======================================================")
print(f"Total Cleaned & Merged Classes: {len(all_classes)}")
print("=======================================================")
for idx, cls in enumerate(all_classes):
    cnt = len(list((DATASET_MERGED_DIR / cls).glob('*.jpg')))
    print(f"{idx+1:02d}. {cls} ({cnt} images)")

## 4. Class Balancing & Data Augmentation
To ensure underrepresented classes have sufficient data, we apply Albumentations image augmentations (Rotation, Flipping, Lighting shifts, Perspective adjustments) up to a target baseline.

In [ ]:
import cv2
import albumentations as A

# Augmentation pipeline
balance_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.4),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=30, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
])

TARGET_MIN_SAMPLES = 250
print(f"Balancing all classes to at least {TARGET_MIN_SAMPLES} images...")

for cls_name in all_classes:
    cls_dir = DATASET_MERGED_DIR / cls_name
    existing_imgs = list(cls_dir.glob("*.jpg"))
    current_count = len(existing_imgs)
    
    if current_count < TARGET_MIN_SAMPLES:
        needed = TARGET_MIN_SAMPLES - current_count
        for i in range(needed):
            src_img_path = existing_imgs[i % current_count]
            img = cv2.imread(str(src_img_path))
            if img is None:
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            augmented = balance_aug(image=img_rgb)['image']
            aug_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
            out_path = cls_dir / f"aug_{i:04d}_{src_img_path.name}"
            cv2.imwrite(str(out_path), aug_bgr)
        print(f"  Augmented {cls_name}: {current_count} -> {len(list(cls_dir.glob('*.jpg')))}")

print("Dataset balancing complete!")

## 5. PyTorch Data Loaders & Transforms
We split the merged dataset into **80% Training**, **10% Validation**, and **10% Test** splits.

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

# Normalization standard for ImageNet
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load full dataset
full_dataset = datasets.ImageFolder(str(DATASET_MERGED_DIR))
class_to_idx = full_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
num_classes = len(class_to_idx)

total_count = len(full_dataset)
train_count = int(0.80 * total_count)
val_count = int(0.10 * total_count)
test_count = total_count - train_count - val_count

generator = torch.Generator().manual_seed(42)
train_sub, val_sub, test_sub = random_split(full_dataset, [train_count, val_count, test_count], generator=generator)

class TransformedDataset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label

train_data = TransformedDataset(train_sub, train_transform)
val_data = TransformedDataset(val_sub, eval_transform)
test_data = TransformedDataset(test_sub, eval_transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Total Images: {total_count}")
print(f"Train Samples: {len(train_data)} | Val Samples: {len(val_data)} | Test Samples: {len(test_data)}")
print(f"Number of Classes: {num_classes}")

## 6. EfficientNet Architecture & Mixed Precision Training
We instantiate an `efficientnet_b4` (or `efficientnet_lite0`) backbone with transfer learning, fine-tuning with Cosine Annealing learning rate schedule.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create Model using timm or torchvision
MODEL_NAME = "efficientnet_b4"
print(f"Instantiating pretrained {MODEL_NAME}...")
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=num_classes, drop_rate=0.3)
model = model.to(device)

# Hyperparameters
EPOCHS = 15
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print("Model and Optimizer configured successfully.")

In [ ]:
from tqdm import tqdm

best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
OUTPUT_DIR = Path("/content/output_model")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Starting Training for {EPOCHS} Epochs on {device}...")
start_time = time.time()

for epoch in range(EPOCHS):
    # --- Training Phase ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data).item()
        total_train += labels.size(0)
        train_bar.set_postfix({'loss': f"{loss.item():.3f}"})
        
    train_loss = running_loss / total_train
    train_acc = (correct_train / total_train) * 100.0
    
    # --- Validation Phase ---
    model.eval()
    val_loss_running, correct_val, total_val = 0.0, 0, 0
    
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False)
    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            val_loss_running += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data).item()
            total_val += labels.size(0)
            
    val_loss = val_loss_running / total_val
    val_acc = (correct_val / total_val) * 100.0
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")
    
    # Save checkpoint if best val accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        checkpoint_payload = {
            'model_state_dict': model.state_dict(),
            'architecture': MODEL_NAME,
            'class_names': idx_to_class,
            'class_to_idx': class_to_idx,
            'num_classes': num_classes,
            'val_acc': val_acc,
            'epoch': epoch + 1,
            'trained_at': time.strftime('%Y-%m-%d %H:%M:%S')
        }
        torch.save(checkpoint_payload, OUTPUT_DIR / "disease_model_best.pth")
        print(f"  --> Best model checkpoint saved! (Val Acc: {val_acc:.2f}%)")

elapsed = (time.time() - start_time) / 60
print(f"\nTraining complete in {elapsed:.1f} minutes. Best Validation Accuracy: {best_val_acc:.2f}%")

## 7. Model Evaluation on Test Set & Visualization

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss', color='darkgreen')
plt.plot(history['val_loss'], label='Val Loss', color='orange')
plt.title('Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Accuracy', color='darkgreen')
plt.plot(history['val_acc'], label='Val Accuracy', color='orange')
plt.title('Accuracy Curves (%)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy %')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=300)
plt.show()

# Evaluate best model on Held-out Test Set
best_ckpt = torch.load(OUTPUT_DIR / "disease_model_best.pth", map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()

test_correct_top1 = 0
test_correct_top3 = 0
total_test = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(images)
        
        # Top 1
        _, preds = outputs.topk(1, dim=1)
        test_correct_top1 += torch.sum(preds.squeeze(1) == labels).item()
        
        # Top 3
        _, preds3 = outputs.topk(3, dim=1)
        test_correct_top3 += torch.sum(preds3 == labels.view(-1, 1)).item()
        
        total_test += labels.size(0)

top1_acc = (test_correct_top1 / total_test) * 100.0
top3_acc = (test_correct_top3 / total_test) * 100.0

print(f"\nHeld-out Test Set Performance ({total_test} images):")
print(f"  Top-1 Accuracy: {top1_acc:.2f}%")
print(f"  Top-3 Accuracy: {top3_acc:.2f}%")

## 8. Export Model Artifacts & `class_names.json`

In [ ]:
# Save class_names.json in the exact format AgriSetu backend consumes
class_names_json_path = OUTPUT_DIR / "class_names.json"
formatted_classes = {str(k): v for k, v in idx_to_class.items()}
with open(class_names_json_path, "w") as f:
    json.dump(formatted_classes, f, indent=2)

print(f"Saved class_names.json with {len(formatted_classes)} classes:")
print(json.dumps(dict(list(formatted_classes.items())[:8]), indent=2))
print("... [truncated]")

## 9. Upload Model to Supabase Storage (Free Tier)

Enter your Supabase URL and Service Role Key (from your `.env`) to upload `disease_model_best.pth` and `class_names.json` directly into your `disease-models` bucket and register the version in `model_versions`.

In [ ]:
from supabase import create_client
import getpass

print("Enter your Supabase Credentials to upload weights:")
SUPABASE_URL = input("Supabase Project URL (e.g. https://xyz.supabase.co): ").strip()
SUPABASE_SERVICE_KEY = getpass.getpass("Supabase Service Role Key (hidden input): ").strip()

if SUPABASE_URL and SUPABASE_SERVICE_KEY:
    try:
        supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
        
        # Upload disease_model_best.pth
        print("Uploading disease_model_best.pth to 'disease-models' bucket...")
        with open(OUTPUT_DIR / "disease_model_best.pth", "rb") as f:
            supabase.storage.from_("disease-models").upload(
                "v2/disease_model_best.pth",
                f,
                file_options={"content-type": "application/octet-stream", "upsert": "true"}
            )
            
        # Upload class_names.json
        print("Uploading class_names.json to 'disease-models' bucket...")
        with open(OUTPUT_DIR / "class_names.json", "rb") as f:
            supabase.storage.from_("disease-models").upload(
                "v2/class_names.json",
                f,
                file_options={"content-type": "application/json", "upsert": "true"}
            )
            
        # Register in model_versions table
        print("Registering model version in Supabase database...")
        supabase.table("model_versions").insert({
            "model_type": "disease_cnn",
            "version": "2.0.0",
            "accuracy": round(float(top1_acc) / 100.0, 4),
            "f1_score": round(float(top3_acc) / 100.0, 4),
            "classes": formatted_classes,
            "weights_url": "disease-models/v2/disease_model_best.pth",
            "is_active": True,
            "notes": f"50+ class EfficientNet-B4 trained on Google Colab GPU. Top-1: {top1_acc:.2f}%, Top-3: {top3_acc:.2f}%",
        }).execute()
        
        print("\nSUCCESS! Model weights and class metadata successfully uploaded to Supabase Storage.")
    except Exception as e:
        print(f"Upload error: {e}")
else:
    print("Skipping Supabase upload (credentials not entered).")

## 10. Direct Local Download / Google Drive Backup
You can also download the trained model and class files directly to your machine or backup to Google Drive.

In [ ]:
from google.colab import files

print("Downloading artifacts to your local browser...")
files.download(str(OUTPUT_DIR / "class_names.json"))
files.download(str(OUTPUT_DIR / "disease_model_best.pth"))

# Optional: Save copy to Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    gdrive_target = Path('/content/drive/MyDrive/agrisetu/models')
    gdrive_target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(OUTPUT_DIR / "disease_model_best.pth", gdrive_target / "disease_model_best.pth")
    shutil.copy2(OUTPUT_DIR / "class_names.json", gdrive_target / "class_names.json")
    print(f"Backup saved to Google Drive at {gdrive_target}")
except Exception as e:
    print(f"Google Drive backup skipped: {e}")